In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
import pymysql
from datetime import datetime,timedelta
import psycopg2
import json
import urllib.request
import psycopg2
import matplotlib.pyplot as plt

#import warnings
#warnings.filterwarnings("ignore")



In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


In [ ]:
def loc_pings(l):
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
        #print(url)
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(vid)])
        print(len(df))
        
    except Exception as e:
        print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df


In [ ]:
plaza_data=pd.read_sql("select * from plaza_code_lat_lons",engine)
plaza_data['latitude']=plaza_data['latitude'].apply(lambda x : round(x,2))
plaza_data['longitude']=plaza_data['longitude'].apply(lambda x : round(x,2))
plaza_data.dropna(inplace=True)
plaza_data['loc_str'] = plaza_data.apply(lambda row: str(int(row['latitude']*100))+str(int(row['longitude']*100)), axis=1)


In [ ]:
def crossed_plazas(l):
    #if 1>0 :
    try:
        vid=l[0]
        fromtime_main=l[1]
        totime_main=l[2]
        loc=pd.DataFrame()
        for j in range(int((totime_main-fromtime_main)/86400)):
           # print(j)
            fromTime=fromtime_main+j*86400
            toTime=fromtime_main+(j+1)*86400
            try:
                loc_reuse=loc_pings([vid,fromTime,toTime])[['vehicleId','location','time','speed']] #dont call raw pings for more than 1 day per vehicle
                loc=pd.concat([loc,loc_reuse])
            except:
                print("No_data for a day")
                
        try:
            loc_reuse=loc_pings([vid,fromTime+(j+1)*86400,totime_main])[['vehicleId','location','time']]
            loc=pd.concat([loc,loc_reuse])
        except:
            print("integral value of days")
        
        
        if len(loc)>0:
            print("raw pings are there")
        else:
            print("Device in No_info for vehicle_id:{}".format(vid))
            return None
        loc=loc[loc['speed']<20]
        loc[['longitude','latitude']] = pd.DataFrame(loc.location.values.tolist(), index= loc.index)
        loc['latitude']=loc['latitude'].apply(lambda x : round(x,2))
        loc['longitude']=loc['longitude'].apply(lambda x : round(x,2))
        loc['loc_str'] = loc.apply(lambda row: str(int(row['latitude']*100))+str(int(row['longitude']*100)), axis=1)
        crossed_plazas=loc[loc['loc_str'].isin(plaza_data['loc_str'])][['vehicleId','time','loc_str']]
        if len(crossed_plazas)>0:
            print("There are crossed plazas")
            crossed_plazas_unique=pd.DataFrame()
            for i in crossed_plazas['loc_str'].unique():
                unique_plaza=crossed_plazas[crossed_plazas['loc_str']==i]
                unique_plaza.sort_values('time',ascending=False,inplace=True)
                unique_plaza['time_diff']=unique_plaza['time'].shift(1)-unique_plaza['time']
                unique_plaza['time_diff'].iloc[0]=86401
                crossed_plazas_visit=unique_plaza[unique_plaza['time_diff']>3600]
                crossed_plazas_unique=pd.concat([crossed_plazas_unique,crossed_plazas_visit])
        else:
            print("NO CROSSED PLAZAS")
        #crossed_plazas_unique=pd.merge(crossed_plazas_unique,plaza_data[['plaza_code','loc_str']],on='loc_str',how='left') 
        crossed_plazas_unique=pd.DataFrame()
        return crossed_plazas_unique
   
    except Exception as e:
        print("issue with crossed_plazas is : " +str(e))
        return None 
    

In [ ]:
list_of_vehicles=[100001, 100002, 100003]  # sample ids, real vehicle ids removed

In [ ]:
iterable=[[i,int(tm.time()-8*86400),int(tm.time())] for i in list_of_vehicles]

In [ ]:
import multiprocessing 
pool=multiprocessing.Pool()
results=pool.map(crossed_plazas,iterable[2:10])
data=pd.concat(results)
pool.close()
pool.join()

In [ ]:
a=crossed_plazas([100001,tm.time()-8*86400,tm.time()])

In [ ]:
a

In [ ]:
plaza_cost=pd.read_sql("select * from plaza_class_cost",engine)

In [ ]:
plaza_cost[plaza_cost['plaza_code']=='189001']